In [3]:
#!pip install langchain transformers pypdf faiss-cpu sentence-transformers
#!pip install langchain_community
#!pip install langchain_huggingface

In [4]:
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.llms import HuggingFacePipeline
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM

# Step 1: Load PDF as LangChain Documents
def load_pdf_as_documents(pdf_path):
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    return documents

# Step 2: Split Documents into Chunks
def split_documents_into_chunks(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    split_docs = text_splitter.split_documents(documents)
    return split_docs

# Step 3: Create FAISS Vector Store
def create_faiss_index_from_documents(documents):
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vector_store = FAISS.from_documents(documents, embeddings)
    return vector_store

# Step 4: Set Up HuggingFace Question Generation
def setup_qg_pipeline():
    model_name = "google/flan-t5-small"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    return pipeline("text2text-generation", model=model, tokenizer=tokenizer)

# Step 5: RAG Pipeline Setup
def setup_rag_pipeline_with_documents(pdf_path):
    # Load PDF and split into chunks
    documents = load_pdf_as_documents(pdf_path)
    split_docs = split_documents_into_chunks(documents)

    # Create vector store
    vector_store = create_faiss_index_from_documents(split_docs)

    # Set up QA pipeline
    retriever = vector_store.as_retriever()
    qg_pipeline = setup_qg_pipeline()
    llm = HuggingFacePipeline(pipeline=qg_pipeline)
    qa_chain = RetrievalQA(llm=llm, retriever=retriever)

    return qa_chain



**Step 1** - Loading PDFs as LangChain Documents
Explanation:

This step uses PyPDFLoader to load a PDF and convert it into a list of Document objects.
Each Document contains text and optional metadata, such as page numbers.

In [6]:
from langchain.document_loaders import PyPDFLoader

# Step 1: Load PDF as LangChain Documents
def load_pdf_as_documents(pdf_path):
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    return documents

# Example: Load a PDF
pdf_path = "example.pdf"  # Replace with your PDF file
documents = load_pdf_as_documents(pdf_path)

# Display the first few documents
print("Number of documents loaded:", len(documents))
print("First document content:", documents[0].page_content[:500])  # Print first 500 characters


Number of documents loaded: 61
First document content: natural-resources.canada.ca /our-natural-resources/energy-sources-distribution/electricity-infrastru…
Powering Canada’s Future: A Clean Electricity
Strategy
155-197 minutes
Table of Contents
Foreword – Clean Electricity Strategy
1.0 The Case for Clean Electricity
1.1 Laying Out a Clean Electricity Strategy for Canada
1.2 A Strategy informed by extensive engagement, electricity sector experts,
and Indigenous energy leaders
1.3 Key Guiding Principles
2.0 Toward the Grid of the Future
2.1 Global Co


**Step 2** - Splitting Documents into Chunks
Explanation:

Large documents need to be divided into smaller chunks to improve retrieval performance.
RecursiveCharacterTextSplitter ensures that the text is split into manageable sizes while maintaining some overlap for context continuity.

In [8]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Step 2: Split Documents into Chunks
def split_documents_into_chunks(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    split_docs = text_splitter.split_documents(documents)
    return split_docs

# Example: Split the loaded documents
chunked_documents = split_documents_into_chunks(documents)
print("Number of chunks created:", len(chunked_documents))
print("First chunk content:", chunked_documents[0].page_content[:500])  # Print first 500 characters


Number of chunks created: 198
First chunk content: natural-resources.canada.ca /our-natural-resources/energy-sources-distribution/electricity-infrastru…
Powering Canada’s Future: A Clean Electricity
Strategy
155-197 minutes
Table of Contents
Foreword – Clean Electricity Strategy
1.0 The Case for Clean Electricity
1.1 Laying Out a Clean Electricity Strategy for Canada
1.2 A Strategy informed by extensive engagement, electricity sector experts,
and Indigenous energy leaders
1.3 Key Guiding Principles
2.0 Toward the Grid of the Future
2.1 Global Co


**Step 3** - Creating a FAISS Vector Store
Explanation:

Text chunks are embedded into numerical vectors using sentence-transformers.
FAISS (Facebook AI Similarity Search) is used to index these vectors for fast retrieval during queries.

In [10]:

from langchain_huggingface import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS

# Step 3: Create FAISS Vector Store
def create_faiss_index_from_documents(documents):
    #embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vector_store = FAISS.from_documents(documents, embeddings)
    vector_store.save_local("faiss_store")
    return vector_store

# Example: Create a FAISS index
vector_store = create_faiss_index_from_documents(chunked_documents)
print("FAISS vector store created!")


FAISS vector store created!


In [11]:
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.llms import HuggingFacePipeline
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM

# Step 4: Set Up HuggingFace Question Generation
def setup_qg_pipeline():
    model_name = "google/flan-t5-small"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    return pipeline("text2text-generation", model=model, tokenizer=tokenizer)

def setup_rag_pipeline_with_documents(pdf_path):
    # Load and process documents
    documents = load_pdf_as_documents(pdf_path)
    split_docs = split_documents_into_chunks(documents)
    vector_store = create_faiss_index_from_documents(split_docs)

    # Set up retriever
    retriever = vector_store.as_retriever()

    # Set up HuggingFacePipeline
    qg_pipeline = setup_qg_pipeline()
    llm = HuggingFacePipeline(pipeline=qg_pipeline)

    # Define a prompt template for QA
    prompt_template = PromptTemplate(
        input_variables=["context", "question"],
        template="Given the context: {context}, answer the question: {question}",
    )

    # Create RetrievalQA chain
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",  # Default chain type for combining documents
        retriever=retriever,
        return_source_documents=True,  # To get the source of the answer
        chain_type_kwargs={"prompt": prompt_template},
    )

    # Return the qa_chain
    return qa_chain  # Added this return statement

# Example: Set up the RAG pipeline
pdf_path = "example.pdf"  # Replace with your PDF file
qa_chain = setup_rag_pipeline_with_documents(pdf_path)

# Query the pipeline
query = "What is the document about?"  # Modify this query as needed
response = qa_chain.invoke(query)

print("\n--- Query Response ---")
print(response)


tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

C:\Users\wayne\anaconda3\envs\hfqa\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\wayne\.cache\huggingface\hub\models--google--flan-t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP dow

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

...

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Device set to use cpu
C:\Users\wayne\AppData\Local\Temp\ipykernel_35148\2403113700.py:24: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=qg_pipeline)
Token indices sequence length is longer than the specified maximum sequence length for this model (601 > 512). Running this sequence through the model will result in indexing errors



--- Query Response ---
{'query': 'What is the document about?', 'result': 'The Kinship and Prosperity Report focuses on six key themes: easing access to funding; developing consistent project eligibility criteria that prioritize Indigenous community benefits; advancing inclusive opportunities and a Just Transition; accelerating Indigenous leadership in the energy transition; respecting self-determination by prioritizing Indigenous-led decisions; and sustainably funding Indigenous participation', 'source_documents': [Document(id='66b03ab4-231d-4282-bbcb-8cf92dcd6219', metadata={'producer': 'Skia/PDF m128', 'creator': 'Chromium', 'creationdate': '2025-01-22T20:42:49+00:00', 'title': 'Powering Canada’s Future: A Clean Electricity Strategy', 'moddate': '2025-01-22T20:42:49+00:00', 'source': 'example.pdf', 'total_pages': 61, 'page': 8, 'page_label': '9'}, page_content='advisory council, which first met in December 2022.\nThe Kinship and Prosperity Report focuses on six key themes: easing a